# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — ML Appendix, Feature Importance for Health Score (Random Forest).**
Average Position (43%), Impressions (32%), and Scroll Depth (15%) top the importance ranking.
Health Score is defined earlier in the paper as Impressions (30 pts) + Position (30 pts) +
CTR (20 pts) + Scroll Depth (20 pts) -- a deterministic weighted sum of three of these same
four inputs. The paper is upfront that "high importance is therefore expected and does not
imply external causation," which is exactly the right disclosure. The methodology question
worth asking, respectfully: since three of the top features are literal components of the
label, what does this ranking look like restricted to features NOT used to build Health
Score -- content age, word count, days since update, search volume, CPC, competition? That
version would show whether anything genuinely external predicts health, rather than mostly
re-deriving the known formula.

**Finding 2 — ML Appendix, Growth & Classification (Logistic Regression, 71% holdout accuracy).**
The paper reports "71% holdout accuracy" without stating the base rate. Finding #1's own
counts give a majority-class base rate: 74.8K growing vs. 45.6K declining pages, so roughly
62% of pages are "growing." Skill point, near verbatim: accuracy of 71% on a label that is
62% positive is 9 points of skill, not 71 -- worth stating next to the headline number, not
instead of it. Second, connected question: was the reported 80/20 holdout split random by
page, or grouped by brand? With 57 brands in the portfolio, a random split risks the model
learning brand-specific patterns and being evaluated on pages from a brand it already saw --
the same failure mode Section 2 below demonstrates directly on our own model.

In [1]:
# Base rate check for Finding 2, from the paper's own Finding #1 counts.
growing = 74_800
declining = 45_600
base_rate = growing / (growing + declining)
reported_accuracy = 0.71

print(f"Majority-class base rate (growing pages): {base_rate:.1%}")
print(f"Reported holdout accuracy: {reported_accuracy:.1%}")
print(f"Real lift over the base rate: {(reported_accuracy - base_rate) * 100:.1f} points")

Majority-class base rate (growing pages): 62.1%
Reported holdout accuracy: 71.0%
Real lift over the base rate: 8.9 points


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

w05 used a grouped-by-client split from the start -- there was no dishonest "before" version
in that notebook to compare against. So the "before" here is a naive random row split (the
thing this whole exercise warns against); the "after" is the same grouped split already used
in w05. Same features, same target, same model, same seed -- only the split changes.

**Result: random split test R^2 = 0.622, grouped split test R^2 = 0.596 -- a real but modest
gap of 0.026.** 44 clients appear in both train and test under the random split; zero overlap
under the grouped split. The random split's score is inflated by partial memorization of
client-specific patterns, though the effect here is smaller than the leakage collapse in
Section 3 -- worth reporting honestly rather than either dismissing or overstating it.

In [2]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

month_03 = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

# --- Same page-month aggregation, position fix, and spike_day_share as w04/w05 ---
page_month = month_03.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
    sessions_organic=("sessions_organic", "sum"),
)

valid_position_rows = month_03[month_03["gsc_avg_position"] > 0].copy()
valid_position_rows["_weight"] = valid_position_rows["gsc_impressions"].clip(lower=1)
valid_position_rows["_weighted_pos"] = valid_position_rows["gsc_avg_position"] * valid_position_rows["_weight"]
grouped_sums = valid_position_rows.groupby(
    ["client_hash_id", "content_hash_id"], as_index=False
)[["_weighted_pos", "_weight"]].sum()
grouped_sums["avg_position"] = grouped_sums["_weighted_pos"] / grouped_sums["_weight"]

page_month = page_month.merge(
    grouped_sums[["client_hash_id", "content_hash_id", "avg_position"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)
page_month = page_month[page_month["avg_position"].notna()].copy()

page_month["ctr_pct"] = np.where(
    page_month["total_impressions"] > 0,
    page_month["total_clicks"] / page_month["total_impressions"] * 100,
    np.nan
)

month_03["day_median"] = month_03.groupby(
    ["client_hash_id", "content_hash_id"]
)["gsc_impressions"].transform("median").clip(lower=1)
month_03["is_spike_day"] = month_03["gsc_impressions"] > 5 * month_03["day_median"]
spike_share_all = month_03.groupby(
    ["client_hash_id", "content_hash_id"]
)[["gsc_impressions", "is_spike_day"]].apply(
    lambda g: g.loc[g["is_spike_day"], "gsc_impressions"].sum() / g["gsc_impressions"].sum()
).reset_index(name="spike_day_share")
page_month = page_month.merge(spike_share_all, on=["client_hash_id", "content_hash_id"], how="left")
page_month["spike_day_share"] = page_month["spike_day_share"].fillna(0.0)

page_month["position_tier"] = pd.cut(
    page_month["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
MIN_IMPRESSIONS = 500
page_month["visible"] = (page_month["total_impressions"] >= MIN_IMPRESSIONS).astype(int)
tier_baseline = page_month[page_month["visible"] == 1].groupby(
    "position_tier", observed=True
)["ctr_pct"].median()
expected_ctr = page_month["position_tier"].astype(str).map(tier_baseline.to_dict()).astype(float)
page_month["ctr_gap"] = expected_ctr - page_month["ctr_pct"]

FEATURES = ["avg_position", "total_impressions", "spike_day_share",
            "ga4_engaged_sessions", "sessions_organic"]
TARGET = "ctr_gap"
model_data = page_month.dropna(subset=FEATURES + [TARGET]).copy()

print("model_data shape:", model_data.shape)

def evaluate_split(train_df, test_df, label):
    X_train, y_train = train_df[FEATURES], train_df[TARGET]
    X_test, y_test = test_df[FEATURES], test_df[TARGET]
    model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    overlap_clients = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
    print(f"{label}: test R^2 = {test_r2:.3f} | "
          f"client overlap between train and test = {len(overlap_clients)}")
    return test_r2

# --- BEFORE: naive random row split, ignoring client groups entirely ---
train_random, test_random = train_test_split(
    model_data, test_size=0.2, random_state=RANDOM_SEED
)
r2_random = evaluate_split(train_random, test_random, "Random split (dishonest)")

# --- AFTER: grouped by client, same as w05 -- zero client overlap by construction ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(model_data, groups=model_data["client_hash_id"]))
train_grouped = model_data.iloc[train_idx]
test_grouped = model_data.iloc[test_idx]
r2_grouped = evaluate_split(train_grouped, test_grouped, "Grouped split (honest)")

print(f"\nGap (random - grouped): {r2_random - r2_grouped:.3f}")
print("A positive gap here means the random split let the model partly memorize")
print("client-specific patterns, and its test score was inflated by seeing other")
print("pages from the same clients during training.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

model_data shape: (175304, 12)
Random split (dishonest): test R^2 = 0.622 | client overlap between train and test = 44
Grouped split (honest): test R^2 = 0.596 | client overlap between train and test = 0

Gap (random - grouped): 0.026
A positive gap here means the random split let the model partly memorize
client-specific patterns, and its test score was inflated by seeing other
pages from the same clients during training.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The attack checklist, applied to this model's actual feature set
(`avg_position`, `total_impressions`, `spike_day_share`, `ga4_engaged_sessions`,
`sessions_organic` predicting `ctr_gap`):

- [x] **Timeline drawn**: all five features are March aggregates; `ctr_gap` is also computed
  from March data, but from `ctr_pct`/`total_clicks` -- columns deliberately excluded from the
  feature set. w05's forward test (April outcome) additionally confirms nothing here peeks
  into a later window.
- [x] **No label-derived or sibling columns**: `total_clicks` and `ctr_pct` build `ctr_gap`
  directly and are excluded from `FEATURES`. Confirmed below: adding `total_clicks` back in
  collapses test R^2 from 0.997 to 0.622 when removed -- a textbook leakage signature, and
  the 0.622 matches Section 2's random-split number exactly.
- [x] **No product flags**: no FlyRank `health_score`/`priority_score`-style column exists in
  this dataset or this feature set.
- [x] **Grouped split**: confirmed in Section 2 -- test R^2 drops from 0.622 (random, 44
  overlapping clients) to 0.596 (grouped, 0 overlap).
- [x] **Base rate printed**: w05's comparison table included the random-ranking base rate
  (0.002) next to every precision@K number; Section 1 above adds the paper's missing base rate.
- [x] **Top feature importance sanity-checked**: `sessions_organic`'s importance in w05 (1.07)
  was investigated and found to be outlier-driven (43x its own 99th percentile), not a
  genuinely dominant signal -- flagged, not celebrated.
- [x] **Out-of-fold metrics**: all w05 and Section 2 numbers are test-set scores, never
  in-sample training scores.

In [3]:
# Deliberately add a label-derived column and watch the score jump toward suspiciously good --
# then remove it and keep the honest number. Same demonstration as w03, run again on this
# model's actual final feature set (per the skill's own verification instruction).
leaky_features = FEATURES + ["total_clicks"]
model_data_leaky = page_month.dropna(subset=leaky_features + [TARGET]).copy()

train_leaky, test_leaky = train_test_split(model_data_leaky, test_size=0.2, random_state=RANDOM_SEED)

leaky_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=RANDOM_SEED)
leaky_model.fit(train_leaky[leaky_features], train_leaky[TARGET])
leaky_r2 = leaky_model.score(test_leaky[leaky_features], test_leaky[TARGET])

honest_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=RANDOM_SEED)
honest_model.fit(train_leaky[FEATURES], train_leaky[TARGET])
honest_r2 = honest_model.score(test_leaky[FEATURES], test_leaky[TARGET])

print(f"WITH total_clicks (leaky): test R^2 = {leaky_r2:.3f}")
print(f"WITHOUT total_clicks (honest): test R^2 = {honest_r2:.3f}")
print(f"Collapse when the leak is removed: {leaky_r2 - honest_r2:.3f}")
print("A large collapse here is the confession the skill describes -- total_clicks (and ctr_pct)")
print("must stay out of the feature set for exactly this reason.")

WITH total_clicks (leaky): test R^2 = 0.997
WITHOUT total_clicks (honest): test R^2 = 0.622
Collapse when the leak is removed: 0.375
A large collapse here is the confession the skill describes -- total_clicks (and ctr_pct)
must stay out of the feature set for exactly this reason.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from w05):** "The baseline wins decisively."

**Rewrite:** On this held-out slice of March 2026 data, evaluated against April's actual
outcome at precision@30, the rule-based baseline outperformed both models by a wide margin
(0.433 vs. 0.100 and 0.000). This is a directional, decision-support finding on one month of
one lane's data -- not a general proof that rule-based scoring beats structural forecasting,
and not evidence about how either method would perform on a different month, a different
lane, or with March's own current-period signal made available to the models.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.